In [ ]:
!pip install evaluate
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.6 MB/s eta 0:00:00


In [ ]:
import os
import torch
import random
import numpy as np
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset
from sacrebleu import corpus_bleu  # Direct import instead of evaluate
from tqdm import tqdm
import kagglehub

# Download the dataset
print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("parvmodi/english-to-malayalam-machine-translation-dataset")
print(f"Dataset downloaded to: {dataset_path}")

eng_file = os.path.join(dataset_path, "train.en")
mal_file = os.path.join(dataset_path, "train.ml")

def load_and_clean_data(eng_file, mal_file, max_samples=2000):
    print(f"Loading first {max_samples} sentence pairs...")
    with open(eng_file, 'r', encoding='utf-8') as f:
        english_raw = [line.strip() for line in f.readlines()][:max_samples]
    with open(mal_file, 'r', encoding='utf-8') as f:
        malayalam_raw = [line.strip() for line in f.readlines()][:max_samples]

    valid_pairs = []
    for en, ml in zip(english_raw, malayalam_raw):
        if en.strip() and ml.strip():
            valid_pairs.append((en, ml))

    english = [pair[0] for pair in valid_pairs]
    malayalam = [pair[1] for pair in valid_pairs]

    print(f"Loaded and cleaned {len(english)} valid parallel sentence pairs")
    return english, malayalam

english, malayalam = load_and_clean_data(eng_file, mal_file)

# Split data
random.seed(42)
indices = list(range(len(english)))
random.shuffle(indices)

train_size = 1200
eval_size = 200
test_size = 200

train_en = [english[i] for i in indices[:train_size]]
train_ml = [malayalam[i] for i in indices[:train_size]]
eval_en = [english[i] for i in indices[train_size:train_size+eval_size]]
eval_ml = [malayalam[i] for i in indices[train_size:train_size+eval_size]]
test_en = [english[i] for i in indices[train_size+eval_size:train_size+eval_size+test_size]]
test_ml = [malayalam[i] for i in indices[train_size+eval_size:train_size+eval_size+test_size]]
mono_ml = [malayalam[i] for i in indices[train_size+eval_size+test_size:]]

print(f"Data splits: Train={len(train_en)}, Eval={len(eval_en)}, Test={len(test_en)}, Mono={len(mono_ml)}")

# Initialize model and tokenizer
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def back_translate(texts, batch_size=8):
    valid_texts = [text for text in texts if text and text.strip()]
    print(f"Back-translating {len(valid_texts)} valid monolingual sentences")

    tokenizer.src_lang = "ml_IN"
    pseudo_english = []
    model.eval()

    for i in tqdm(range(0, len(valid_texts), batch_size), desc="Back-translating"):
        batch_texts = valid_texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True,
                         truncation=True, max_length=128).to(device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"],
                max_length=128,
                num_beams=4
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        pseudo_english.extend(decoded)

    return pseudo_english

print("Performing back-translation...")
pseudo_english = back_translate(mono_ml)

augmented_train_en = [en for en in (train_en + pseudo_english) if en is not None and en.strip()]
augmented_train_ml = [ml for ml in (train_ml + mono_ml) if ml is not None and ml.strip()]
min_len = min(len(augmented_train_en), len(augmented_train_ml))
augmented_train_en = augmented_train_en[:min_len]
augmented_train_ml = augmented_train_ml[:min_len]

print(f"Augmented training data size after cleaning: {len(augmented_train_en)}")

def prepare_dataset(en_texts, ml_texts, max_length=128):
    valid_en = [text for text in en_texts if text is not None and text.strip()]
    valid_ml = [text for text in ml_texts if text is not None and ml.strip()]
    min_len = min(len(valid_en), len(valid_ml))
    valid_en = valid_en[:min_len]
    valid_ml = valid_ml[:min_len]

    data_dict = {"en": valid_en, "ml": valid_ml}
    dataset = Dataset.from_dict(data_dict)

    def preprocess_function(examples):
        inputs = tokenizer(examples["en"], max_length=max_length, truncation=True, padding="max_length")
        labels = tokenizer(examples["ml"], max_length=max_length, truncation=True, padding="max_length")
        inputs["labels"] = labels["input_ids"]
        return inputs

    return dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

train_dataset = prepare_dataset(augmented_train_en, augmented_train_ml)
eval_dataset = prepare_dataset(eval_en, eval_ml)

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Use sacrebleu directly
    result = corpus_bleu(decoded_preds, [decoded_labels])
    return {"bleu": result.score}

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./mbart_finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_steps=100,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

def evaluate_test_set(test_english, test_malayalam):
    valid_pairs = [(en, ml) for en, ml in zip(test_english, test_malayalam) if en.strip() and ml.strip()]
    valid_en = [pair[0] for pair in valid_pairs]
    valid_ml = [pair[1] for pair in valid_pairs]

    model.eval()
    tokenizer.src_lang = "en_XX"
    inputs = tokenizer(valid_en, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id["ml_IN"],
            max_length=128,
            num_beams=4
        )

    predictions = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    # Use sacrebleu directly
    bleu_score = corpus_bleu(predictions, [valid_ml]).score
    return bleu_score, predictions

test_bleu, test_predictions = evaluate_test_set(test_en, test_ml)
print(f"Test BLEU Score: {test_bleu:.2f}")

trainer.save_model("./mbart_finetuned")
tokenizer.save_pretrained("./mbart_finetuned")
print("Model saved to ./mbart_finetuned")

In [ ]:
for name in list(globals().keys()):
    if not name.startswith("_"):
        del globals()[name]

import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
%reset -f
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
!pip install transformers sacrebleu kagglehub

import os
import random
import torch
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration
from sacrebleu import corpus_bleu
import kagglehub

# Download dataset if needed
dataset_path = kagglehub.dataset_download("parvmodi/english-to-malayalam-machine-translation-dataset")
eng_file = os.path.join(dataset_path, "train.en")
mal_file = os.path.join(dataset_path, "train.ml")

def load_and_clean_data(eng_file, mal_file, max_samples=2000):
    with open(eng_file, 'r', encoding='utf-8') as f:
        english_raw = [line.strip() for line in f][:max_samples]
    with open(mal_file, 'r', encoding='utf-8') as f:
        malayalam_raw = [line.strip() for line in f][:max_samples]
    valid_pairs = [(en, ml) for en, ml in zip(english_raw, malayalam_raw) if en and ml]
    english = [pair[0] for pair in valid_pairs]
    malayalam = [pair[1] for pair in valid_pairs]
    return english, malayalam

english, malayalam = load_and_clean_data(eng_file, mal_file)
random.seed(42)
indices = list(range(len(english)))
random.shuffle(indices)
test_size = 20  # Fewer for quick run!
test_en = [english[i] for i in indices[:test_size]]
test_ml = [malayalam[i] for i in indices[:test_size]]

# Load model/tokenizer if present
model_name = "facebook/mbart-large-50-many-to-many-mmt"
try:
    tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
    model = MBartForConditionalGeneration.from_pretrained(model_name)
    print("Model loaded successfully.")
except Exception as e:
    print("Could not load mBART model:", e)
    raise

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

tokenizer.src_lang = "en_XX"
inputs = tokenizer(test_en, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
with torch.no_grad():
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["ml_IN"],
        max_length=128,
        num_beams=4
    )

predictions = tokenizer.batch_decode(generated, skip_special_tokens=True)
bleu = corpus_bleu(predictions, [test_ml])

report_lines = []
report_lines.append("mBART Model Evaluation on Test Samples")
report_lines.append("===================================")
report_lines.append(f"Test samples: {test_size}")
report_lines.append(f"Corpus BLEU Score: {bleu.score:.2f}\n")
report_lines.append("Sample Translations:")
for idx, (src, pred, ref) in enumerate(zip(test_en, predictions, test_ml)):
    report_lines.append(f"{idx+1}. English: {src}")
    report_lines.append(f"   Malayalam Prediction: {pred}")
    report_lines.append(f"   Reference: {ref}\n")

with open("model_working_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"Report saved to model_working_report.txt")

# OPTIONAL: If you want to download from Colab UI, run this line:
# from google.colab import files
# files.download("model_working_report.txt")

Using Colab cache for faster access to the 'english-to-malayalam-machine-translation-dataset' dataset.
Model loaded successfully.
Report saved to model_working_report.txt


In [ ]:
!pip install transformers sacrebleu kagglehub --quiet

import os, random, torch
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration
from sacrebleu import corpus_bleu, corpus_chrf
import kagglehub

# Load dataset
eng_path = kagglehub.dataset_download("parvmodi/english-to-malayalam-machine-translation-dataset") + "/train.en"
mal_path = kagglehub.dataset_download("parvmodi/english-to-malayalam-machine-translation-dataset") + "/train.ml"
with open(eng_path, encoding="utf-8") as f: english = [l.strip() for l in f][:200]
with open(mal_path, encoding="utf-8") as f: malayalam = [l.strip() for l in f][:200]
pairs = [(en, ml) for en, ml in zip(english, malayalam) if en and ml]
test_en, test_ml = zip(*pairs)

# Load model
tok = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()
tok.src_lang = "en_XX"

# Batch inference
predictions = []
batch_size = 8
for i in range(0, len(test_en), batch_size):
    inputs = tok(list(test_en[i:i+batch_size]), return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, forced_bos_token_id=tok.lang_code_to_id["ml_IN"], max_length=128, num_beams=4)
    predictions += tok.batch_decode(out, skip_special_tokens=True)

# BLEU, chrF, report
bleu = corpus_bleu(predictions, [list(test_ml)])
chrf = corpus_chrf(predictions, [list(test_ml)])
report = [
    f"Test sentences: {len(test_en)}",
    f"BLEU: {bleu.score:.2f}",
    f"chrF: {chrf.score:.2f}",
    f"Brevity Penalty: {bleu.bp:.3f}",
    "Sample translations:"
]
for i in range(5):
    report += [f"{i+1}. EN: {test_en[i]}\nML: {predictions[i]}\nREF: {test_ml[i]}\n"]

with open("mbart_report.txt", "w", encoding="utf-8") as f: f.write('\n'.join(report))
from google.colab import files; files.download("mbart_report.txt")

Using Colab cache for faster access to the 'english-to-malayalam-machine-translation-dataset' dataset.
Using Colab cache for faster access to the 'english-to-malayalam-machine-translation-dataset' dataset.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>